# Bases de Datos con Python
## Conexión y consultas con SQLite

***Python permite trabajar con sistemas gestores de bases de datos como SQLite, MySQL o PostgreSQL***. Es decir, desde un programa podemos ejecutar instrucciones SQL para almacenar, consultar, modificar y eliminar información.

En estos apuntes, vamos a ver cómo conectarse a una base de datos ***SQLite*** desde Python y realizar operaciones SQL: crear tablas, insertar, consultar, actualizar y eliminar registros.
Vemos SQLite porque:
- No necesita un servidor independiente.
- Guarda los datos en un archivo.
- Permite trabajar con SQL.
- Se integra con Python mediante `sqlite3`.





### Objetivos
- Conectarse a una base de datos.
- Crear tablas e insertar registros con  `CREATE TABLE` y `INSERT INTO`.
- Ejecutar consultas `SELECT`.
- Actualizar o modificar registros con `UPDATE`. 
- Recuperar resultados con `fetchone()` y `fetchall()`.
- Eliminar registros usando `DELETE`.



## 2. Conexión a una base de datos


### 2.1. Importar el módulo y conectarse

La instrucción `sqlite3.connect()` establece una conexión. Si el archivo no existe, SQLite lo crea. En estos ejemplos utilizaremos `./BBDD/tienda.db`, que se guardará en el subdirectorio `./BBDD`.


In [15]:
import sqlite3

conexion = sqlite3.connect("./BBDD/tienda.db")
print("Conexión establecida")

conexion.close()

Conexión establecida


### 2.2. Crear una tabla

Un *cursor* es un objeto que permite ejecutar instrucciones SQL sobre una base de datos y recuperar los resultados de las consultas. Un cursor permite ejecutar sentencias SQL. Hacemos un `commit()` para confirmar los cambios y `close()` cierra la conexión.

Vamos a crear una tabla `productos` que tendrá cuatro columnas: `id` (identificador), `nombre`, `precio` y `stock`.


In [20]:
import sqlite3

conexion = sqlite3.connect("./BBDD/tienda.db")
cursor = conexion.cursor()

cursor.execute("""
    CREATE TABLE IF NOT EXISTS productos (
        id INTEGER PRIMARY KEY,
        nombre TEXT NOT NULL,
        precio REAL,
        stock INTEGER
    )
""")

conexion.commit()
conexion.close()

print("Tabla productos creada (si no existía).")

Tabla productos creada (si no existía).


## 3. Insertar datos

### 3.1. Insertar un registro

La instrucción `INSERT INTO` añade un registro a una tabla.


In [23]:
conexion = sqlite3.connect("./BBDD/tienda.db")
cursor = conexion.cursor()

cursor.execute("""
    INSERT INTO productos (nombre, precio, stock)
    VALUES ('Teclado', 25.50, 10)
""")

conexion.commit()
conexion.close()

**Atención:** si ejecutas varias veces la celda anterior, se insertará el mismo producto repetidamente.

### 3.2. Insertar datos mediante parámetros

Es preferible utilizar `?` como marcador de posición y pasar los valores por separado. Así evitamos construir consultas concatenando cadenas y reducimos el riesgo de inyección SQL.


In [26]:
conexion = sqlite3.connect("./BBDD/tienda.db")
cursor = conexion.cursor()

nombre = "Ratón"
precio = 15.90
stock = 20

cursor.execute("""
    INSERT INTO productos (nombre, precio, stock)
    VALUES (?, ?, ?)
""", (nombre, precio, stock))

conexion.commit()
conexion.close()

## 4. Consultar datos desde Python

La instrucción `SELECT` recupera información. Para ejecutar la consulta usamos `execute()` y para recoger resultados podemos usar `fetchone()`, `fetchall()` o `fetchmany()`.

### 4.1. Consultar todos los registros


In [30]:
conexion = sqlite3.connect("./BBDD/tienda.db")
cursor = conexion.cursor()

cursor.execute("SELECT * FROM productos")
productos = cursor.fetchall()

for producto in productos:
    print(producto)

conexion.close()

(1, 'Teclado', 25.5, 10)
(2, 'Ratón', 15.9, 20)


Cada registro se devuelve como una tupla de Python, por ejemplo: `(1, 'Teclado', 25.5, 10)`.

### 4.2. Consultar determinadas columnas

Podemos indicar solo las columnas que queremos recuperar.


In [33]:
conexion = sqlite3.connect("./BBDD/tienda.db")
cursor = conexion.cursor()

cursor.execute("SELECT nombre, precio FROM productos")
productos = cursor.fetchall()

for producto in productos:
    print(producto)

conexion.close()

('Teclado', 25.5)
('Ratón', 15.9)


### 4.3. Consultar con una condición: `WHERE`

La cláusula `WHERE` selecciona los registros que cumplen una condición. En este ejemplo, consultamos productos con precio inferior a 20 euros.


In [36]:
conexion = sqlite3.connect("./BBDD/tienda.db")
cursor = conexion.cursor()

cursor.execute("""
    SELECT nombre, precio
    FROM productos
    WHERE precio < ?
""", (20,))

for producto in cursor.fetchall():
    print(producto)

conexion.close()

('Ratón', 15.9)


**Parámetros de una consulta:** el valor se pasa como una tupla. Cuando hay un único elemento, se escribe una coma: `(20,)`.

### 4.4. Recuperar un registro con `fetchone()`

`fetchone()` devuelve el siguiente registro disponible. Si no hay resultados, devuelve `None`.


In [39]:
conexion = sqlite3.connect("./BBDD/tienda.db")
cursor = conexion.cursor()

cursor.execute("""
    SELECT nombre, precio
    FROM productos
    WHERE id = ?
""", (1,))

producto = cursor.fetchone()
print(producto)

conexion.close()

('Teclado', 25.5)


### 4.5. Diferencias entre los métodos de recuperación

| Método | Descripción |
|---|---|
| `fetchone()` | Recupera un registro. |
| `fetchall()` | Recupera todos los registros restantes. |
| `fetchmany(n)` | Recupera hasta `n` registros. |


## 5. Modificar y eliminar registros

### 5.1. Actualizar con `UPDATE`

`UPDATE` modifica los registros que cumplen la condición `WHERE`. Si se omite `WHERE`, se modificarán todos los registros de la tabla.


In [43]:
conexion = sqlite3.connect("./BBDD/tienda.db")
cursor = conexion.cursor()

cursor.execute("""
    UPDATE productos
    SET precio = ?
    WHERE id = ?
""", (29.90, 1))

conexion.commit()
conexion.close()

### 5.2. Eliminar con `DELETE`

`DELETE` elimina los registros que cumplen la condición. Si se omite `WHERE`, se eliminarán todos los registros de la tabla. Comprueba siempre la condición antes de ejecutar la consulta.


In [46]:
conexion = sqlite3.connect("./BBDD/tienda.db")
cursor = conexion.cursor()

cursor.execute("""
    DELETE FROM productos
    WHERE id = ?
""", (1,))

conexion.commit()
conexion.close()

## 6. Resumen de métodos

| Método | Función |
|---|---|
| `sqlite3.connect()` | Establece una conexión. |
| `conexion.cursor()` | Crea un cursor. |
| `cursor.execute()` | Ejecuta una sentencia SQL. |
| `cursor.executemany()` | Ejecuta una sentencia con varios conjuntos de parámetros. |
| `cursor.fetchone()` | Recupera un registro. |
| `cursor.fetchall()` | Recupera los registros restantes. |
| `cursor.fetchmany(n)` | Recupera hasta `n` registros. |
| `conexion.commit()` | Confirma los cambios. |
| `conexion.close()` | Cierra la conexión. |

### Esquema general de trabajo
1. Importar `sqlite3`.
2. Conectarse a la base de datos.
3. Crear un cursor.
4. Ejecutar sentencias SQL.
5. Recuperar resultados, si procede.
6. Confirmar los cambios cuando corresponda.
7. Cerrar la conexión.


## 7. Ejercicio práctico

La base de datos `tienda.db` contiene una tabla `productos` con esta información:

| id | nombre | precio | stock |
|---:|---|---:|---:|
| 1 | Teclado | 25.50 | 10 |
| 2 | Ratón | 15.90 | 20 |
| 3 | Monitor | 180.00 | 5 |
| 4 | Webcam | 35.00 | 8 |
| 5 | Auriculares | 45.00 | 12 |

Escribe un programa en Python que realice las siguientes operaciones:

a. Conectarse a `tienda.db`.  
b. Mostrar todos los productos.  
c. Mostrar solo el nombre y el precio.  
d. Mostrar los productos con precio superior a 30 euros.  
e. Consultar el producto cuyo identificador introduzca el usuario.  
f. Insertar un producto solicitando nombre, precio y stock.  
g. Actualizar el precio de un producto a partir de su identificador.  
h. Eliminar un producto a partir de su identificador.  
i. Mostrar los productos con stock inferior a 10 unidades.  
j. Mostrar los productos ordenados por precio de menor a mayor.


## 8. Buenas prácticas y ampliación

- Usa consultas parametrizadas (`?`) en lugar de concatenar valores en el SQL.
- Confirma los cambios con `commit()` cuando insertes, actualices o elimines datos.
- Cierra las conexiones cuando termines.
- Comprueba cuidadosamente las condiciones `WHERE` de las consultas `UPDATE` y `DELETE`.
- En programas más completos, gestiona errores con `try`/`except` y utiliza `rollback()` si una transacción falla.

### Próximos contenidos
1. Consultas con `ORDER BY`, `GROUP BY` y funciones de agregación.
2. Gestión de errores y transacciones.
3. Conexión a MySQL o PostgreSQL.


## CSV y bases de datos

Un archivo CSV es un formato sencillo para almacenar e intercambiar datos tabulares. Una base de datos, en cambio, permite gestionar los datos mediante un ***sistema gestor*** que proporciona herramientas para consultarlos, modificarlos, relacionarlos y mantener su integridad.
CSV es especialmente útil para intercambiar y analizar datos, mientras que, una base de datos es especialmente útil para gestionar información estructurada dentro de una aplicación.


Podemos cargar los datos de un archivo CSV en una tabla de una base de datos utilizando Python. Es una operación muy habitual cuando queremos importar datos para después consultarlos mediante SQL.

In [55]:
import pandas as pd
import sqlite3

# 1. Leer el archivo CSV
productos = pd.read_csv("./BBDD/ventas.csv")

# 2. Conectar con la base de datos
conexion = sqlite3.connect("tienda.db")

# 3. Cargar los datos en una tabla
productos.to_sql(
    "ventas",   # Nombre de la tabla que queremos crear o utilizar.
    conexion,
    if_exists = "replace",
    index=False    # Evita guardar el índice del DataFrame como una columna adicional.
)

cursor = conexion.cursor()

cursor.execute("SELECT * FROM productos")



# 4. Cerrar la conexión
conexion.close()

print("Datos cargados correctamente")

OperationalError: no such table: productos

La instrucción `if_exists = replace` elimina la tabla existente y la vuelve a crear. Otras opciones son `fail`que genera un error si la tabla ya existe y `append` que añade registros a la tabla existente.